# Concatenation, Schema Alignment & Production Data Integration

In [2]:
import pandas as pd
import numpy as np

jan_sales = pd.DataFrame({
    'order_id': [1001, 1002, 1003],
    'customer_id': [101, 102, 103],
    'sales': [2500, 1800, 3200],
    'city': ['Delhi', 'Mumbai', 'Delhi']
})

feb_sales = pd.DataFrame({
    'order_id': [1004, 1005, 1006],
    'customer_id': [104, 105, 101],
    'sales': [1500, 4200, 2800],
    'region': ['North', 'West', 'North']
})

mar_sales = pd.DataFrame({
    'order_id': [1007, 1008, 1009],
    'customer_id': [102, 106, 103],
    'sales': [2100, 3500, 1900],
    'city': ['Mumbai', 'Jaipur', 'Delhi'],
    'discount': [10, 5, 0]
})

correction_data = pd.DataFrame({
    'order_id': [1002, 1005],
    'customer_id': [102, 105],
    'sales': [2000, 4000],
    'city': ['Mumbai', 'Mumbai']
})

#### Q1. Basic row concatenation

Combine jan_sales, feb_sales, and mar_sales vertically using pd.concat(). Keep all columns and inspect the resulting missing values.

In [4]:
result=pd.concat([jan_sales,feb_sales,mar_sales])
result

,order_id,customer_id,sales,city,region,discount
0,1001,101,2500,Delhi,NaN,NaN
1,1002,102,1800,Mumbai,NaN,NaN
2,1003,103,3200,Delhi,NaN,NaN
0,1004,104,1500,NaN,North,NaN
1,1005,105,4200,NaN,West,NaN
2,1006,101,2800,NaN,North,NaN
0,1007,102,2100,Mumbai,NaN,10.0
1,1008,106,3500,Jaipur,NaN,5.0
2,1009,103,1900,Delhi,NaN,0.0


#### Q2. Reset the index

Repeat Q1 with a continuous integer index. Explain why ignore_index=True is useful when combining monthly datasets.

In [6]:
result=pd.concat([jan_sales,feb_sales,mar_sales],ignore_index=True)
result

,order_id,customer_id,sales,city,region,discount
0,1001,101,2500,Delhi,NaN,NaN
1,1002,102,1800,Mumbai,NaN,NaN
2,1003,103,3200,Delhi,NaN,NaN
3,1004,104,1500,NaN,North,NaN
4,1005,105,4200,NaN,West,NaN
5,1006,101,2800,NaN,North,NaN
6,1007,102,2100,Mumbai,NaN,10.0
7,1008,106,3500,Jaipur,NaN,5.0
8,1009,103,1900,Delhi,NaN,0.0


#### Q3. Schema alignment

Create a standardized dataset containing order_id, customer_id, sales, city, region, and discount. Fill unavailable fields with appropriate missing values rather than inventing data.

In [8]:
import pandas as pd
import numpy as np

# Standard column schema
required_columns = [
    'order_id',
    'customer_id',
    'sales',
    'city',
    'region',
    'discount'
]

# Standardize the existing DataFrame
result = result.reindex(columns=required_columns)

# Preserve missing values; do not invent replacements
result['order_id'] = result['order_id'].astype('Int64')
result['customer_id'] = result['customer_id'].astype('Int64')
result['sales'] = pd.to_numeric(result['sales'], errors='coerce')
result['discount'] = pd.to_numeric(result['discount'], errors='coerce')

# Display the standardized dataset
display(result)

# Verify schema and missing values
print("Column order:", result.columns.tolist())
print("\nData types:")
print(result.dtypes)

print("\nMissing values:")
print(result.isna().sum())

,order_id,customer_id,sales,city,region,discount
0,1001,101,2500,Delhi,NaN,NaN
1,1002,102,1800,Mumbai,NaN,NaN
2,1003,103,3200,Delhi,NaN,NaN
3,1004,104,1500,NaN,North,NaN
4,1005,105,4200,NaN,West,NaN
5,1006,101,2800,NaN,North,NaN
6,1007,102,2100,Mumbai,NaN,10.0
7,1008,106,3500,Jaipur,NaN,5.0
8,1009,103,1900,Delhi,NaN,0.0


Column order: ['order_id', 'customer_id', 'sales', 'city', 'region', 'discount']

Data types:
order_id         Int64
customer_id      Int64
sales            int64
city            object
region          object
discount       float64
dtype: object

Missing values:
order_id       0
customer_id    0
sales          0
city           3
region         6
discount       6
dtype: int64


#### Q4. Track data sources

Concatenate the three monthly datasets while adding a source_month column to each before combining them. Ensure every row retains its month of origin.

In [43]:
jan_sales['source_month'] = 'January'
feb_sales['source_month'] = 'February'
mar_sales['source_month'] = 'March'
first_quarter=pd.concat([jan_sales,feb_sales,mar_sales]).reset_index()
first_quarter

,index,order_id,customer_id,sales,city,source_month,region,discount
0,0,1001,101,2500,Delhi,January,NaN,NaN
1,1,1002,102,1800,Mumbai,January,NaN,NaN
2,2,1003,103,3200,Delhi,January,NaN,NaN
3,0,1004,104,1500,NaN,February,North,NaN
4,1,1005,105,4200,NaN,February,West,NaN
5,2,1006,101,2800,NaN,February,North,NaN
6,0,1007,102,2100,Mumbai,March,NaN,10.0
7,1,1008,106,3500,Jaipur,March,NaN,5.0
8,2,1009,103,1900,Delhi,March,NaN,0.0


#### Q5. Column-wise concatenation

Create two DataFrames with the same index:  Use pd.concat(axis=1) to combine them. Explain how pandas aligns rows.

In [27]:
customer_info = pd.DataFrame({
    'customer_name': ['Aman', 'Riya', 'Kabir']
}, index=[101, 102, 103])

customer_city = pd.DataFrame({
    'city': ['Delhi', 'Mumbai', 'Delhi']
}, index=[101, 102, 103])

result=pd.concat([customer_info,customer_city],axis=1)
result

,customer_name,city
101,Aman,Delhi
102,Riya,Mumbai
103,Kabir,Delhi


#### Q6. Detect duplicate orders

Combine the monthly datasets and identify duplicated order_id values. Return all occurrences of duplicate orders.

In [38]:
first_quarter[first_quarter['order_id'].duplicated(keep=False)]

,order_id,customer_id,sales,city,source_month,region,discount


#### Q7. Apply correction records

Use correction_data to update matching orders in the combined sales dataset. Preserve orders that have no correction record. Do not accidentally duplicate corrected orders.

In [41]:
correction_data

,order_id,customer_id,sales,city
0,1002,102,2000,Mumbai
1,1005,105,4000,Mumbai


In [45]:
first_quarter = first_quarter.set_index('order_id')
correction_data = correction_data.set_index('order_id')

first_quarter.update(correction_data)

first_quarter = first_quarter.reset_index()

display(first_quarter)

,order_id,index,customer_id,sales,city,source_month,region,discount
0,1001,0,101,2500,Delhi,January,NaN,NaN
1,1002,1,102,2000,Mumbai,January,NaN,NaN
2,1003,2,103,3200,Delhi,January,NaN,NaN
3,1004,0,104,1500,NaN,February,North,NaN
4,1005,1,105,4000,Mumbai,February,West,NaN
5,1006,2,101,2800,NaN,February,North,NaN
6,1007,0,102,2100,Mumbai,March,NaN,10.0
7,1008,1,106,3500,Jaipur,March,NaN,5.0
8,1009,2,103,1900,Delhi,March,NaN,0.0


#### Q8. Validate data types

After concatenation, inspect the data types of every column. Standardize order_id and customer_id to integer types and explain how missing values can affect integer columns.

In [48]:
first_quarter.dtypes

order_id          int64
index             int64
customer_id       int64
sales             int64
city             object
source_month     object
region           object
discount        float64
dtype: object

#### Q9. Build a monthly sales summary

After standardizing and combining the monthly datasets, calculate:
- Number of orders
- Total sales
- Average sales
- Unique customers
Group the report by source_month.

In [53]:
report=first_quarter.groupby('source_month').agg(
    No_of_orders=('order_id','count'),
    total_sales=('sales','sum'),
    avg_sales=('sales','mean'),
    unique_customers=('customer_id','nunique')
)
report

,No_of_orders,total_sales,avg_sales,unique_customers
source_month,,,,
February,3,8300,2766.666667,3
January,3,7700,2566.666667,3
March,3,7500,2500.000000,3


#### Q10. Production integration pipeline

Create a reusable function named combine_sales_data() that:
1. Accepts a list of monthly DataFrames.
2. Adds a source identifier to each dataset.
3. Aligns their schemas.
4. Concatenates them with a fresh index.
5. Reports duplicate order IDs.
6. Returns the combined DataFrame.